# TrustLens Day 2 — Deepfake Classifier (EfficientNet-B0) on Colab

Transfer-learn EfficientNet-B0 on **GenImage**, holding out **one generator
family** as an out-of-distribution (OOD) set. The headline result is the
**OOD accuracy drop** — the distribution-shift exhibit.

Runtime: **Runtime → Change runtime type → GPU (T4)**. Budget ~2–3 h on a
GenImage subset (capped via `max_per_class_per_gen`).

Pipeline: frozen backbone → train head → unfreeze last blocks → fine-tune,
all tracked in **MLflow**.

## 0. GPU check

In [ ]:
!nvidia-smi -L
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1. Get the TrustLens code + install deps
Torch/torchvision ship with Colab; we add MLflow and install the package.

In [ ]:
# Clone the repo (or replace with your fork)
![ -d TrustLens ] || git clone https://github.com/umerjavaidkh/TrustLens.git
%cd TrustLens
!pip -q install mlflow pyngrok
!pip -q install -e .   # installs trustlens (numpy, pillow, piexif, etc.)
import sys; sys.path.insert(0, "src")

## 2. Get GenImage data

GenImage is large (multi-GB per generator). Put a **subset** where Colab can see
it. Two common options:

**A. Google Drive** — upload a `genimage/` folder, then mount:
```
from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/genimage'
```

**B. Kaggle** — set your kaggle.json, then pull a generator subset:
```
!pip -q install kaggle
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d <genimage-mirror-slug> -p /content/genimage --unzip
DATA_ROOT = '/content/genimage'
```

Expected layout (label folders `ai`=fake, `nature`=real):
```
genimage/<generator>/{train,val}/{ai,nature}/*.png
```

In [ ]:
# Set this to your GenImage root:
DATA_ROOT = "/content/genimage"

from trustlens.models.splits import index_dataset, list_generators
records = index_dataset(DATA_ROOT)
print(f"Indexed {len(records)} labelled images")
print("Generators found:", list_generators(records))

## 3. Configure the OOD hold-out

Pick one generator family to hold out. Train on the rest, evaluate on it. A GAN
family (e.g. `BigGAN`) held out while training on diffusion models is a strong
shift; choose based on what your `list_generators` printed above.

In [ ]:
from trustlens.training.train import TrainConfig

cfg = TrainConfig(
    data_root=DATA_ROOT,
    ood_generator="BigGAN",       # <-- must match a name printed above
    img_size=224,
    batch_size=64,
    head_epochs=3,                # Phase 1: frozen backbone
    finetune_epochs=5,            # Phase 2: unfreeze last blocks
    unfreeze_blocks=2,
    max_per_class_per_gen=2000,   # cap to fit the ~2-3h budget; raise if you have time
    target_fpr=0.01,
    experiment="trustlens-deepfake",
    out_dir="models",
)
cfg

## 4. Train (two-phase) + evaluate ID vs OOD
MLflow logs params, per-epoch metrics, the best checkpoint, and the OOD drop.

In [ ]:
from trustlens.training.train import train
model, report, logs = train(cfg)

## 5. The distribution-shift exhibit

In [ ]:
import json
print(report.summary_line())
print(json.dumps(report.as_dict(), indent=2))

# Plot in-distribution vs OOD
import matplotlib.pyplot as plt
labels = ["ID test", f"OOD\n({report.ood_generator})"]
accs = [report.id_test.accuracy, report.ood.accuracy]
plt.bar(labels, accs, color=["#2a9d8f", "#e76f51"])
plt.ylim(0, 1); plt.ylabel("accuracy")
plt.title(f"OOD accuracy drop = {report.accuracy_drop:.3f} ({report.relative_drop:.0%})")
for i, a in enumerate(accs): plt.text(i, a+0.01, f"{a:.3f}", ha="center")
plt.show()

## 6. Inspect the MLflow run
View metrics inline, or tunnel the MLflow UI with ngrok (needs an ngrok token).

In [ ]:
import mlflow
run = mlflow.search_runs(experiment_names=[cfg.experiment]).iloc[0]
cols = [c for c in run.index if c.startswith("metrics.")]
print(run[cols])

# Optional live UI:
# from pyngrok import ngrok
# get_ipython().system_raw('mlflow ui --port 5000 &')
# print("MLflow UI:", ngrok.connect(5000))

## 7. Save the best model to Drive

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# import shutil; shutil.copy('models/efficientnet_b0_best.pt',
#     '/content/drive/MyDrive/trustlens_efficientnet_b0_best.pt')
print('Best checkpoint:', 'models/efficientnet_b0_best.pt')

## Notes

- **Why one family out?** In-distribution accuracy overstates real-world
  performance because fraudsters use *new* generators. The OOD drop quantifies
  how much the detector degrades on an unseen generator — the number that
  matters operationally, and the one Day-3 work will try to shrink.
- **Time budget:** `max_per_class_per_gen` caps images per (generator, class).
  Start at 2000; if a phase is fast and you have GPU time, raise it or add epochs.
- **Light augmentation on purpose:** heavy geometric/JPEG augmentation can erase
  the subtle high-frequency fingerprints the model relies on.